In [1]:
import os
import sys
import pickle
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")
script_name = source_path+"/scripts/deep_feature_extraction.py"

In [3]:
model_list=global_vars.list_models()
df_pre_patch, df_pre_extracted,_,_ =file_IO.preprocess_experiment_logs(source_path)
viz_numeric=file_IO.get_reference_table_for_experiments(df_pre_patch,df_pre_extracted,im_show=False,im_plot=False)

In [4]:
squares_standard_name = 'squares_gw5.0_m5.0_idx2' #'rectangles_gw3.0_m5.0_idx1' #'squares_gw5.0_m5.0_idx2'
body_standard_name = 'body_gwnan_m1.0_idx1'
print(file_IO.get_experiment_from_unique_name(df_pre_patch, squares_standard_name))
print(file_IO.get_experiment_from_unique_name(df_pre_patch, body_standard_name))

icdar_train_df_patches_20250515_164130.csv
icdar_train_df_body_20250523_181312.csv


In [15]:
save_to_log=False #if true the extracted outputs are saved as csv in the preprocessed folder. If false
#they are saved i the specific model folders 
dataset_name=body_standard_name#squares_standard_name #body_standard_name #squares_standard_name
already_used_models=file_IO.get_models_applied_to_unique_name(viz_numeric, dataset_name)
print("Already used models on ",dataset_name,": ",already_used_models)
verbose=False #if verbose the batch per batch extraction info is printed

modality='test' #'test'
if save_to_log==False:
    input_preprocessed=file_IO.load_preprocessed_files(kind='body', mode=modality) #patches_224, patches_standard, body
else:
    input_preprocessed=file_IO.get_experiment_from_unique_name(df_pre_patch, dataset_name)
    print("Input preprocessed: ", input_preprocessed)
extra_view=False
contrastive_mode = False  # Set to True for contrastive learning
custom_pretrained = 'original'  # 'original', 'contrastive', 'fine-tune'
data_augmentation = False  # Set to True for data augmentation
is_train = modality #val, test, train
if is_train == 'val':
    zarr='test_public_writers'
elif is_train == 'test':
    zarr='test_private_writers'
else:
    zarr='train_writers'

n_patches=-1 #40 #with negative values all patches are considered
num_views = 1  # Number of views for data augmentation, set to 1 for no augmentation

Already used models on  body_gwnan_m1.0_idx1 :  ['trocr-small-stage1', 'trocr-small-handwritten', 'resnet50', 'vit-base-patch16-224-in21k', 'trocr-base-handwritten', 'clip-vit-large-patch14', 'trocr-large-handwritten', 'dresnet50', 'crnn_vgg16_bn', 'vitstr_base', 'trocr-large-stage1', 'trocr-base-stage1', 'alexnet', 'vgg16', 'googlenet', 'resnet18', 'DeiT-Tiny', 'swin_b', 'swin_s', 'DeiT-Small', 'DeiT-Small-Dist', 'DeiT-Base', 'BEiT-Base', 'clip-vit-base-patch16', 'clip-vit-base-patch32', 'clip-vit-large-patch14-un', 'vit-base-patch16-224', 'vit-base-patch32-224-in21k', 'vit-large-patch16-224-in21k', 'vit-huge-patch14-224-in21k', 'vitstr_small', 'db_mobilenet', 'crnn_mobilenet_224', 'linknet_resnet50_224', 'linknet_resnet18', 'crnn_mobilenet', 'sar_resnet31', 'alexnet_gap', 'densenet161', 'densenet121', 'densenet201', 'vgg11', 'vgg16_512', 'maxvit', 'efficientnet_v2_l', 'efficientnet_v2_s', 'convnext_large', 'convnext_base', 'convnext_small', 'mobilenet_v3_small', 'mobilenet_v3_large',

In [16]:
#redo=['vgg16']
selected_models =  [x for x in model_list if x not in already_used_models]
selected_models = ['resnet34_layer2','resnet34','DeiT-Tiny-inter','DeiT-Tiny','crnn_vgg16_bn_224-inter','crnn_vgg16_bn_224',
                   'dresnet50-inter','dresnet50','convnext_large-inter','convnext_large','trocr-large-handwritten-inter','trocr-large-handwritten',
                   'clip-vit-large-patch14-inter','clip-vit-large-patch14-un','BEiT-Large-inter','BEiT-Large'] 
selected_models = ['clip-vit-large-patch14-inter'] 
huggingface_parameter = [global_vars.get_props(name).hugging for name in selected_models]

In [17]:
print("Selected models for extraction: ",selected_models)

Selected models for extraction:  ['clip-vit-large-patch14-inter']


# launch deep_feature_extraction script

In [18]:
for i,model in enumerate(selected_models):
    if save_to_log==False:
        if contrastive_mode:
            base_dir = source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_models[i]}\\contrastive\\extracted_representation\\'
        else:
            base_dir = source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_models[i]}\\representation_extraction\\extracted_representation\\'
        if data_augmentation:
            suffix = '_augmented'
        else:
            suffix=''
        if extra_view:
            save_dir = base_dir+f'extra_view\\{is_train}{suffix}'
        else:
            save_dir = base_dir+f'{is_train}{suffix}'
        file_IO.access_or_create_dir(save_dir)
    else:
        save_dir=None
    print(f"Running model {model} ({i+1} out of {len(selected_models)})")
    args = script_launching.DotDict(
        N_max=282, #useless parameter in the script
        patches=True,
        input_filename=input_preprocessed,
        huggingface=huggingface_parameter[i],
        pooling=False,  # if true in transformer models use pooling, if false only the cls token
        custom_transform=False,
        transform_mode='resize',
        save_h5=False,
        selected_model=model,  # googlenet, alexnet
        truncation='remove head',
        running='new-laptop',
        saved='old-laptop',
        model_mode='truncated',  # 'truncation
        batching=True,
        batch_size=16,#64
        select_cls=False,
        num_workers=4,#4
        pin_memory=True,
        show_image=True,
        save_to_log=save_to_log,  # Save the extracted features to a log file
        data_augmentation=data_augmentation,
        num_views=num_views,  # Number of views for data augmentation
        save_dir=save_dir,
        n_patches=n_patches,
        zarr_path = f"C:\\Users\\andre\PhD\Datasets\ICDAR 2013 - Gender Identification Competition Dataset\\{zarr}.zarr",
        contrastive_mode=contrastive_mode,  # Set to True for contrastive learning
        augmentation_code = 'simple',
        custom_pretrained=custom_pretrained,
        verbose=verbose,
        #script_mode='standalone'
    )
    if save_to_log==False:
        file_IO.save_args(args,save_dir)  # Save the arguments to a file
    script_launching.run_experiment_threaded(args,script_name)  # Test a single run first

Running model clip-vit-large-patch14-inter (1 out of 1)
Starting experiment:
[STDOUT] Saving debug images...
[STDOUT] Debug images saved.
Experiment finished with return code: 0


In [5]:
print(0)

0


# reload

In [14]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    import utils.script_launching as script_launching
    import utils.global_vars as global_vars
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)
    importlib.reload(script_launching)
    importlib.reload(global_vars)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching, global_vars
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching, global_vars = reload_modules()